In [1]:
import os,sys
notebook_dir = os.getcwd()
path = os.path.abspath(os.path.join(notebook_dir, "Code/LightGBM"))
sys.path.append(path)
import lightgbm as lgbm
from Dataset import ModelDataset
from torch.utils.data import DataLoader
import torch
import numpy as np
import pandas as pd
from generateSplits import generateSplits
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

metadata = pd.read_csv("../Datasets/BreastDCEDL_spy1/BreastDCEDL_spy1_metadata.csv")
train_df,val_df = generateSplits(metadata,0.2,seed=42)
train_df = train_df[["pid","pCR","ER","PR","HER2"]].set_index("pid",drop=True)
val_df = val_df[["pid","pCR","ER","PR","HER2"]].set_index("pid",drop=True)
skf = StratifiedKFold(n_splits=4,shuffle=True,random_state=42)

In [3]:
def dataloader_to_Xy(dataloader:torch.utils.data.DataLoader):
    X_list, y_list = [], []
    for img,mols,labels in dataloader:
        img = img.cpu()
        mols = mols.cpu()
        labels = labels.cpu()
        X = np.concatenate([img.view(img.size(0), -1).numpy(),mols.numpy()],axis=1)
        y = labels.numpy()
        X_list.append(X)
        y_list.append(y)
    X = np.concatenate(X_list, axis=0)
    y = np.concatenate(y_list, axis=0)
    return X,y

In [4]:
train_dataset = ModelDataset(train_df,class_samples={0:3,1:8},caching=True,cache_dir=r"E:\SRP\SRP-2025-Project\cache",loading_bar=False)
val_dataset = ModelDataset(val_df,caching=True,cache_dir=r"E:\SRP\SRP-2025-Project\cache",loading_bar=False)
X_train,y_train = dataloader_to_Xy(DataLoader(train_dataset,batch_size=8,shuffle=True))
X_val,y_val = dataloader_to_Xy(DataLoader(val_dataset,batch_size=8,shuffle=False))

Dataset initialised with 542 entries.
Dataset initialised with 32 entries.


In [ ]:
clf = lgbm.LGBMClassifier(n_estimators=20,device="cpu",early_stopping_rounds=5)
clf.fit(X_train, y_train,eval_set=[(X_train, y_train),(X_val,y_val)],eval_metric="auc")

[LightGBM] [Warning] early_stopping_round is set=5, early_stopping_rounds=5 will be ignored. Current value: early_stopping_round=5
[LightGBM] [Info] Number of positive: 272, number of negative: 270
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 47.382288 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 142349299
[LightGBM] [Info] Number of data points in the train set: 542, number of used features: 786435
[LightGBM] [Warning] early_stopping_round is set=5, early_stopping_rounds=5 will be ignored. Current value: early_stopping_round=5
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.501845 -> initscore=0.007380
[LightGBM] [Info] Start training from score 0.007380
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 5 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further 